<a href="https://colab.research.google.com/github/Rds1007/SQL_BigDataInterview/blob/main/gaps_and_islands.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("PandasToSpark").getOrCreate()
df=spark.read.csv("/content/sample_data/sf_events_sample.csv",header=True,inferSchema=True)
#df.show()

In [44]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

In [45]:
window=Window.partitionBy('user_id').orderBy('record_date')

In [46]:
df = df.withColumn("rn",row_number().over(window))

In [47]:
df=df.withColumn("diff",df.record_date-df.rn)

In [48]:
from pyspark.sql.functions.builtin import col
df=df.groupBy('user_id','diff').count()

In [50]:
df=df.filter(col('count')>=3)

In [51]:
df.show()

+-------+----------+-----+
|user_id|      diff|count|
+-------+----------+-----+
|     U4|2021-01-08|    3|
+-------+----------+-----+



In [ ]:
#Gaps and islands questions

WITH cte AS (
    SELECT
        user_id,
        record_date,
        record_date -
            ROW_NUMBER() OVER (
                PARTITION BY user_id
                ORDER BY record_date
            ) * INTERVAL '1 day' AS grp
    FROM sf_events
)
SELECT user_id
FROM cte
GROUP BY user_id, grp
HAVING COUNT(*) >= 3;
